# Stage 2: RSNA External-Dataset Evaluation

## Research Objective

This stage evaluates the **Stage 1 PneumoniaMNIST CNN** on the external **RSNA Pneumonia Detection Challenge** dataset.

The purpose is to test whether the model's predictions and confidence behaviour remain similar when the model is exposed to chest X-ray images from a different dataset.

This is an **external-dataset evaluation**. It is not the later cross-hospital MIMIC-CXR to CheXpert experiment described in the wider research plan.

### Main questions

1. How does the Stage 1 model behave on the RSNA dataset?
2. How well does it separate the two RSNA study-level classes?
3. What probability scores does it produce?
4. Are the confidence scores suitable for the calibration analysis in Stage 3?

### Reproducibility

The raw RSNA DICOM files remain outside the GitHub repository.

This notebook only works with the local dataset paths and saves the derived prediction file needed for Stage 3.


## 1. Set up the environment

We start by installing and importing the small number of packages needed for reading the RSNA DICOM files, preparing the images, loading the Stage 1 CNN, and saving the predictions.

The paths below follow the directory structure already used in the project.


In [27]:
%pip install pydicom


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [28]:
# Import the standard libraries first.
# Path is used for Windows paths because it is easier to read and maintain
# than repeatedly joining strings by hand.
import json
from pathlib import Path
from collections import Counter, defaultdict

# pandas is used to create the study-level and DICOM-level tables.
import pandas as pd

# pydicom reads the medical images stored in DICOM format.
import pydicom

# NumPy is used for the prediction arrays that will later be passed to Stage 3.
import numpy as np

# PyTorch is used to load the trained Stage 1 model and run inference.
import torch
import torch.nn as nn

# These utilities handle the RSNA image conversion and batching.
from PIL import Image
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader

print("PyTorch version:", torch.__version__)
print("pydicom version:", pydicom.__version__)


PyTorch version: 2.5.1+cpu
pydicom version: 3.0.2


## 2. Define the project paths

Keeping the important paths in one place makes the notebook easier to reproduce and reduces the chance of accidentally reading or saving a file from the wrong location.

The raw RSNA data are read from the existing dataset directory. The trained Stage 1 model is loaded from the project `models` directory, and the final predictions are saved under `research_data`.


In [29]:
# Official RSNA annotation file used to construct the study-level labels.
annotation_path = Path(
    r"D:\0001_Kings_AI_Project\datasets\rsna_pneumonia"
    r"\annotations\pneumonia-challenge-annotations-original_2018.json"
)

# Directory containing the extracted RSNA DICOM images.
dicom_root = Path(
    r"D:\0001_Kings_AI_Project\datasets\rsna_pneumonia"
    r"\raw\images"
)

# Stage 1 model checkpoint.
# This file should be created after training the Stage 1 SmallCNN.
checkpoint_path = Path(
    r"D:\0001_Kings_AI_Project\trustworthy-ai-medical-imaging"
    r"\models\smallcnn_pneumoniamnist_stage1.pt"
)

# Intermediate study-level labels created by this notebook.
study_labels_path = Path(
    r"D:\0001_Kings_AI_Project\datasets\rsna_pneumonia"
    r"\rsna_study_level_labels.csv"
)

# Final prediction file consumed by Stage 3.
prediction_path = Path(
    r"D:\0001_Kings_AI_Project\trustworthy-ai-medical-imaging"
    r"\research_data\rsna_stage2_predictions.npz"
)

print("Annotation file:", annotation_path)
print("DICOM directory:", dicom_root)
print("Stage 1 checkpoint:", checkpoint_path)
print("Study-level labels:", study_labels_path)
print("Stage 2 predictions:", prediction_path)


Annotation file: D:\0001_Kings_AI_Project\datasets\rsna_pneumonia\annotations\pneumonia-challenge-annotations-original_2018.json
DICOM directory: D:\0001_Kings_AI_Project\datasets\rsna_pneumonia\raw\images
Stage 1 checkpoint: D:\0001_Kings_AI_Project\trustworthy-ai-medical-imaging\models\smallcnn_pneumoniamnist_stage1.pt
Study-level labels: D:\0001_Kings_AI_Project\datasets\rsna_pneumonia\rsna_study_level_labels.csv
Stage 2 predictions: D:\0001_Kings_AI_Project\trustworthy-ai-medical-imaging\research_data\rsna_stage2_predictions.npz


## 3. Check the required files before doing any processing

This cell gives us an early, readable error if one of the required files or directories is missing.

The Stage 1 checkpoint is especially important. Stage 2 must evaluate the **trained Stage 1 model** rather than create a new model with random weights.


In [30]:
# Check the annotation file.
if not annotation_path.exists():
    raise FileNotFoundError(
        f"RSNA annotation file was not found:\n{annotation_path}"
    )

# Check the extracted DICOM directory.
if not dicom_root.exists():
    raise FileNotFoundError(
        f"RSNA DICOM directory was not found:\n{dicom_root}"
    )

# Check the trained Stage 1 model.
if not checkpoint_path.exists():
    raise FileNotFoundError(
        f"Stage 1 model checkpoint was not found:\n{checkpoint_path}\n\n"
        "Run the Stage 1 model-saving cell before starting Stage 2."
    )

print("All required Stage 2 inputs are available.")


All required Stage 2 inputs are available.


## 4. Load and inspect the RSNA annotation file

Before constructing labels, we inspect the annotation structure rather than assuming a particular JSON format.

This is useful because the RSNA annotation file contains nested information about datasets, labels, studies, and annotation records.


In [31]:
# Open the RSNA annotation JSON in read-only mode.
with open(annotation_path, "r", encoding="utf-8") as file:
    data = json.load(file)

print("JSON loaded successfully.")
print("Top-level type:", type(data).__name__)

if isinstance(data, dict):
    print("Top-level keys:", list(data.keys()))

print("Number of datasets:", len(data["datasets"]))
print("Number of studies:", len(data["datasets"][0]["studies"]))
print("Number of annotations:", len(data["datasets"][0]["annotations"]))


JSON loaded successfully.
Top-level type: dict
Top-level keys: ['id', 'createdAt', 'updatedAt', 'name', 'description', 'isPrivate', 'labelGroups', 'datasets']
Number of datasets: 1
Number of studies: 25684
Number of annotations: 28989


In [32]:
# Inspect one annotation record so we can see the actual fields
# used by the RSNA dataset.
first_annotation = data["datasets"][0]["annotations"][0]

print("First annotation record:")
print(first_annotation)

print("\nFields in the record:")
for key, value in first_annotation.items():
    print(f"  {key}: {type(value).__name__}")


First annotation record:
{'id': 'A_47O6aP', 'createdAt': '2018-08-20T18:55:44.967Z', 'createdById': 'U_E8bx9Q', 'updatedAt': '2018-08-20T18:55:44.972Z', 'updatedById': 'U_E8bx9Q', 'machineLearningModelId': None, 'StudyInstanceUID': '1.2.276.0.7230010.3.1.2.8323329.5872.1517874318.729899', 'SeriesInstanceUID': '1.2.276.0.7230010.3.1.3.8323329.5872.1517874318.729898', 'SOPInstanceUID': '1.2.276.0.7230010.3.1.4.8323329.5872.1517874318.729900', 'labelId': 'L_ylR0L8', 'annotationNumber': None, 'height': None, 'width': None, 'data': None, 'note': None, 'radlexTagIds': [], 'reviewsPositiveCount': 0, 'reviewsNegativeCount': 0}

Fields in the record:
  id: str
  createdAt: str
  createdById: str
  updatedAt: str
  updatedById: str
  machineLearningModelId: NoneType
  StudyInstanceUID: str
  SeriesInstanceUID: str
  SOPInstanceUID: str
  labelId: str
  annotationNumber: NoneType
  height: NoneType
  width: NoneType
  data: NoneType
  note: NoneType
  radlexTagIds: list
  reviewsPositiveCount: in

## 5. Understand the RSNA label definitions

The RSNA annotation file defines the labels rather than requiring us to hard-code arbitrary label IDs.

We create two lookup tables:

- `label_map` converts a label ID into its human-readable name.
- `label_details` keeps the additional metadata associated with each label.


In [33]:
# Build a lookup from the RSNA label ID to its human-readable name.
label_map = {}

# Keep the additional properties of each label as well.
label_details = {}

for label in data["labelGroups"][0]["labels"]:

    label_id = label["id"]

    label_map[label_id] = label["name"]

    label_details[label_id] = {
        "name": label["name"],
        "type": label["type"],
        "scope": label["scope"],
        "annotationMode": label["annotationMode"]
    }

print("RSNA label definitions:\n")

for label_id, details in label_details.items():
    print(f"Label ID: {label_id}")
    print(f"  Name: {details['name']}")
    print(f"  Type: {details['type']}")
    print(f"  Scope: {details['scope']}")
    print(f"  Annotation mode: {details['annotationMode']}")
    print()


RSNA label definitions:

Label ID: L_ylR0L8
  Name: No Lung Opacity
  Type: global
  Scope: instance
  Annotation mode: None

Label ID: L_DlqEAl
  Name: Lung Opacity
  Type: local
  Scope: instance
  Annotation mode: bbox



## 6. Validate the annotation distribution

A study can contain more than one annotation record. Before reducing the annotations to one study-level label, we check how these records are distributed.

This gives us a simple audit trail for the label-construction decision.


In [34]:
annotations = data["datasets"][0]["annotations"]

# Count how many annotation records belong to each label.
annotation_label_counts = Counter(
    label_map[annotation["labelId"]]
    for annotation in annotations
)

print("Annotation counts by label:")
for label_name, count in annotation_label_counts.items():
    print(f"  {label_name}: {count}")


# Count annotations per study.
annotations_per_study = Counter(
    annotation["StudyInstanceUID"]
    for annotation in annotations
)

distribution = Counter(
    annotations_per_study.values()
)

print("\nAnnotation-count distribution per study:")

for annotation_count, study_count in sorted(distribution.items()):
    print(
        f"  {annotation_count} annotation(s): "
        f"{study_count} study/studies"
    )

print(
    "\nTotal unique studies with annotations:",
    len(annotations_per_study)
)


Annotation counts by label:
  No Lung Opacity: 20025
  Lung Opacity: 8964

Annotation-count distribution per study:
  1 annotation(s): 22506 study/studies
  2 annotation(s): 3062 study/studies
  3 annotation(s): 105 study/studies
  4 annotation(s): 11 study/studies

Total unique studies with annotations: 25684


In [35]:
# Group all annotation records by StudyInstanceUID.
# This allows us to see which label combinations occur within a study.
study_annotations = defaultdict(list)

for annotation in annotations:
    study_uid = annotation["StudyInstanceUID"]
    study_annotations[study_uid].append(annotation)


# Count the distinct label combinations found within each study.
label_combinations = Counter()

for study_uid, study_annots in study_annotations.items():

    labels_in_study = sorted(
        set(
            label_map[annotation["labelId"]]
            for annotation in study_annots
        )
    )

    combination = " + ".join(labels_in_study)
    label_combinations[combination] += 1


print("Label combinations by study:\n")

for combination, count in label_combinations.most_common():
    print(f"  {combination}: {count}")


Label combinations by study:

  No Lung Opacity: 20025
  Lung Opacity: 5659


## 7. Construct one ground-truth label per study

For the external evaluation, we reduce the annotation records to one binary study-level label:

- `1` = Lung Opacity
- `0` = No Lung Opacity

`StudyInstanceUID` is used as the linking key because it is available both in the RSNA annotation data and in the DICOM metadata.


In [36]:
# Create one row per RSNA study.
# A study is treated as positive when a Lung Opacity annotation is present.
study_rows = []

for study_uid, study_annots in study_annotations.items():

    label_names = {
        label_map[annotation["labelId"]]
        for annotation in study_annots
    }

    if "Lung Opacity" in label_names:

        label = 1
        label_name = "Lung Opacity"

    elif "No Lung Opacity" in label_names:

        label = 0
        label_name = "No Lung Opacity"

    else:
        # Keep unexpected cases out of the evaluation rather than
        # silently assigning them an incorrect class.
        continue

    study_rows.append({
        "StudyInstanceUID": study_uid,
        "label": label,
        "label_name": label_name,
        "annotation_count": len(study_annots)
    })


rsna_labels = pd.DataFrame(study_rows)

print("RSNA study-level dataset created.")
print("Number of studies:", len(rsna_labels))

print("\nLabel distribution:")
print(rsna_labels["label_name"].value_counts())


RSNA study-level dataset created.
Number of studies: 25684

Label distribution:
label_name
No Lung Opacity    20025
Lung Opacity        5659
Name: count, dtype: int64


In [37]:
# Run a few integrity checks before joining the labels to the DICOM files.
print(
    "Duplicate StudyInstanceUIDs:",
    rsna_labels["StudyInstanceUID"].duplicated().sum()
)

print(
    "Missing labels:",
    rsna_labels["label"].isna().sum()
)

print(
    "Unique studies:",
    rsna_labels["StudyInstanceUID"].nunique()
)

print(
    "Total rows:",
    len(rsna_labels)
)


# Save the clean study-level table so the label construction is reproducible.
study_labels_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

rsna_labels.to_csv(
    study_labels_path,
    index=False
)

print("\nSaved study-level labels to:")
print(study_labels_path)


Duplicate StudyInstanceUIDs: 0
Missing labels: 0
Unique studies: 25684
Total rows: 25684

Saved study-level labels to:
D:\0001_Kings_AI_Project\datasets\rsna_pneumonia\rsna_study_level_labels.csv


## 8. Discover the RSNA DICOM files

The official RSNA archive was extracted into the `raw\images` directory.

We search recursively because the DICOM files can be stored inside nested folders.


In [38]:
# Find every DICOM file under the extracted RSNA image directory.
dicom_files = sorted(
    dicom_root.rglob("*.dcm")
)

print("DICOM files found:", len(dicom_files))


DICOM files found: 25684


In [39]:
# Inspect the identifiers from a few DICOM files.
# We only read metadata here, so the image pixels are not loaded yet.

for path in dicom_files[:5]:

    ds = pydicom.dcmread(
        path,
        stop_before_pixels=True
    )

    print("\nFile:", path.name)
    print(
        "StudyInstanceUID:",
        getattr(ds, "StudyInstanceUID", None)
    )
    print(
        "SeriesInstanceUID:",
        getattr(ds, "SeriesInstanceUID", None)
    )
    print(
        "SOPInstanceUID:",
        getattr(ds, "SOPInstanceUID", None)
    )



File: 1.2.276.0.7230010.3.1.4.8323329.10001.1517874346.163716.dcm
StudyInstanceUID: 1.2.276.0.7230010.3.1.2.8323329.10001.1517874346.163715
SeriesInstanceUID: 1.2.276.0.7230010.3.1.3.8323329.10001.1517874346.163714
SOPInstanceUID: 1.2.276.0.7230010.3.1.4.8323329.10001.1517874346.163716

File: 1.2.276.0.7230010.3.1.4.8323329.10002.1517874346.165592.dcm
StudyInstanceUID: 1.2.276.0.7230010.3.1.2.8323329.10002.1517874346.165591
SeriesInstanceUID: 1.2.276.0.7230010.3.1.3.8323329.10002.1517874346.165590
SOPInstanceUID: 1.2.276.0.7230010.3.1.4.8323329.10002.1517874346.165592

File: 1.2.276.0.7230010.3.1.4.8323329.10004.1517874346.174866.dcm
StudyInstanceUID: 1.2.276.0.7230010.3.1.2.8323329.10004.1517874346.174865
SeriesInstanceUID: 1.2.276.0.7230010.3.1.3.8323329.10004.1517874346.174864
SOPInstanceUID: 1.2.276.0.7230010.3.1.4.8323329.10004.1517874346.174866

File: 1.2.276.0.7230010.3.1.4.8323329.10005.1517874346.191950.dcm
StudyInstanceUID: 1.2.276.0.7230010.3.1.2.8323329.10005.1517874346.19

## 9. Connect every DICOM image to its RSNA study label

The annotation file gives us the study-level ground truth, while each DICOM contains its `StudyInstanceUID`.

We therefore build a DICOM-level table by reading the identifiers from each DICOM and joining the corresponding study-level label.

No image pixels are loaded during this step.


In [40]:
# Create a simple lookup from StudyInstanceUID to the binary RSNA label.
label_lookup = dict(
    zip(
        rsna_labels["StudyInstanceUID"],
        rsna_labels["label"]
    )
)


# Store the metadata needed for the external evaluation.
dicom_records = []

for i, path in enumerate(dicom_files, start=1):

    # Read only the DICOM metadata.
    # This keeps this potentially long scan much faster than loading
    # the image pixels for every file.
    ds = pydicom.dcmread(
        path,
        stop_before_pixels=True
    )

    study_uid = getattr(ds, "StudyInstanceUID", None)
    series_uid = getattr(ds, "SeriesInstanceUID", None)
    sop_uid = getattr(ds, "SOPInstanceUID", None)

    dicom_records.append({
        "dicom_path": str(path),
        "StudyInstanceUID": study_uid,
        "SeriesInstanceUID": series_uid,
        "SOPInstanceUID": sop_uid,
        "label": label_lookup.get(study_uid)
    })

    # Print occasional progress so we know the scan is still running.
    if i % 2000 == 0 or i == len(dicom_files):
        print(
            f"Processed {i:,} / {len(dicom_files):,} DICOM files"
        )


dicom_df = pd.DataFrame(dicom_records)

print("\nFinished reading DICOM metadata.")
print("Rows:", len(dicom_df))


Processed 2,000 / 25,684 DICOM files
Processed 4,000 / 25,684 DICOM files
Processed 6,000 / 25,684 DICOM files
Processed 8,000 / 25,684 DICOM files
Processed 10,000 / 25,684 DICOM files
Processed 12,000 / 25,684 DICOM files
Processed 14,000 / 25,684 DICOM files
Processed 16,000 / 25,684 DICOM files
Processed 18,000 / 25,684 DICOM files
Processed 20,000 / 25,684 DICOM files
Processed 22,000 / 25,684 DICOM files
Processed 24,000 / 25,684 DICOM files
Processed 25,684 / 25,684 DICOM files

Finished reading DICOM metadata.
Rows: 25684


In [41]:
# Check whether the DICOM-to-label connection worked as expected.
print("Total DICOM files:", len(dicom_df))
print(
    "Unique StudyInstanceUIDs:",
    dicom_df["StudyInstanceUID"].nunique()
)

print(
    "DICOMs with matching RSNA label:",
    dicom_df["label"].notna().sum()
)

print(
    "DICOMs without matching RSNA label:",
    dicom_df["label"].isna().sum()
)

print(
    "Duplicate SOPInstanceUIDs:",
    dicom_df["SOPInstanceUID"].duplicated().sum()
)

print(
    "Missing StudyInstanceUID:",
    dicom_df["StudyInstanceUID"].isna().sum()
)


Total DICOM files: 25684
Unique StudyInstanceUIDs: 25684
DICOMs with matching RSNA label: 25684
DICOMs without matching RSNA label: 0
Duplicate SOPInstanceUIDs: 0
Missing StudyInstanceUID: 0


## 10. Inspect the image encoding

Before preprocessing thousands of images, we inspect the DICOM image characteristics.

The Stage 1 model expects a single-channel 64 × 64 image with tensor values between 0 and 1. The checks below help us confirm the RSNA images can be converted into that format consistently.


In [42]:
# Inspect the image metadata from a few representative DICOM files.
for path in dicom_files[:5]:

    ds = pydicom.dcmread(
        path,
        stop_before_pixels=False
    )

    print("\nFile:", path.name)
    print("Rows:", getattr(ds, "Rows", None))
    print("Columns:", getattr(ds, "Columns", None))
    print(
        "PhotometricInterpretation:",
        getattr(ds, "PhotometricInterpretation", None)
    )
    print(
        "SamplesPerPixel:",
        getattr(ds, "SamplesPerPixel", None)
    )
    print(
        "BitsAllocated:",
        getattr(ds, "BitsAllocated", None)
    )
    print(
        "BitsStored:",
        getattr(ds, "BitsStored", None)
    )
    print(
        "PixelRepresentation:",
        getattr(ds, "PixelRepresentation", None)
    )
    print(
        "RescaleSlope:",
        getattr(ds, "RescaleSlope", None)
    )
    print(
        "RescaleIntercept:",
        getattr(ds, "RescaleIntercept", None)
    )
    print("Pixel array shape:", ds.pixel_array.shape)



File: 1.2.276.0.7230010.3.1.4.8323329.10001.1517874346.163716.dcm
Rows: 1024
Columns: 1024
PhotometricInterpretation: MONOCHROME2
SamplesPerPixel: 1
BitsAllocated: 8
BitsStored: 8
PixelRepresentation: 0
RescaleSlope: None
RescaleIntercept: None
Pixel array shape: (1024, 1024)

File: 1.2.276.0.7230010.3.1.4.8323329.10002.1517874346.165592.dcm
Rows: 1024
Columns: 1024
PhotometricInterpretation: MONOCHROME2
SamplesPerPixel: 1
BitsAllocated: 8
BitsStored: 8
PixelRepresentation: 0
RescaleSlope: None
RescaleIntercept: None
Pixel array shape: (1024, 1024)

File: 1.2.276.0.7230010.3.1.4.8323329.10004.1517874346.174866.dcm
Rows: 1024
Columns: 1024
PhotometricInterpretation: MONOCHROME2
SamplesPerPixel: 1
BitsAllocated: 8
BitsStored: 8
PixelRepresentation: 0
RescaleSlope: None
RescaleIntercept: None
Pixel array shape: (1024, 1024)

File: 1.2.276.0.7230010.3.1.4.8323329.10005.1517874346.191950.dcm
Rows: 1024
Columns: 1024
PhotometricInterpretation: MONOCHROME2
SamplesPerPixel: 1
BitsAllocated: 8

In [43]:
# Check the image dimensions without loading the pixel data.
dimension_counts = Counter()

for i, path in enumerate(dicom_files, start=1):

    ds = pydicom.dcmread(
        path,
        stop_before_pixels=True
    )

    dimensions = (
        getattr(ds, "Rows", None),
        getattr(ds, "Columns", None)
    )

    dimension_counts[dimensions] += 1

    if i % 2000 == 0 or i == len(dicom_files):
        print(
            f"Checked {i:,} / {len(dicom_files):,}"
        )


print("\nImage dimension distribution:")

for dimensions, count in dimension_counts.most_common():
    print(f"  {dimensions}: {count}")


Checked 2,000 / 25,684
Checked 4,000 / 25,684
Checked 6,000 / 25,684
Checked 8,000 / 25,684
Checked 10,000 / 25,684
Checked 12,000 / 25,684
Checked 14,000 / 25,684
Checked 16,000 / 25,684
Checked 18,000 / 25,684
Checked 20,000 / 25,684
Checked 22,000 / 25,684
Checked 24,000 / 25,684
Checked 25,684 / 25,684

Image dimension distribution:
  (1024, 1024): 25684


## 11. Prepare the RSNA images for the Stage 1 CNN

Stage 1 used 64 × 64 grayscale PneumoniaMNIST images converted with `transforms.ToTensor()`.

For this external evaluation, we keep the preprocessing deliberately simple:

1. Read the DICOM pixel array.
2. Treat the image as a single grayscale channel.
3. Resize it to 64 × 64.
4. Convert it to a PyTorch tensor.

We do not introduce an additional normalization step here because that was not part of the Stage 1 preprocessing.


In [44]:
# This transform matches the basic image-size and tensor conversion
# used by the Stage 1 PneumoniaMNIST experiment.
rsna_transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor()
])


def preprocess_rsna_dicom(path):
    """
    Read one RSNA DICOM image and convert it into the input format
    expected by the Stage 1 CNN.
    """

    # Read the DICOM including its image pixels.
    ds = pydicom.dcmread(path)

    # The inspected RSNA images are single-channel grayscale images.
    pixel_array = ds.pixel_array

    # Convert the NumPy image into a PIL grayscale image.
    image = Image.fromarray(
        pixel_array,
        mode="L"
    )

    # Resize the image and convert it to a tensor with values from 0 to 1.
    tensor = rsna_transform(image)

    return tensor


In [45]:
# Test the preprocessing on one image before processing the full dataset.
# This gives us a quick sanity check that the tensor has the expected shape.

test_path = dicom_files[0]

test_tensor = preprocess_rsna_dicom(
    test_path
)

print("Test image:", test_path.name)
print("Tensor shape:", test_tensor.shape)
print("Tensor dtype:", test_tensor.dtype)
print("Minimum value:", test_tensor.min().item())
print("Maximum value:", test_tensor.max().item())


Test image: 1.2.276.0.7230010.3.1.4.8323329.10001.1517874346.163716.dcm
Tensor shape: torch.Size([1, 64, 64])
Tensor dtype: torch.float32
Minimum value: 0.0
Maximum value: 0.8117647171020508


## 12. Recreate the Stage 1 CNN architecture

The external evaluation must use the same architecture that was trained in Stage 1.

The model is intentionally small. The purpose of this experiment is not to build a state-of-the-art pneumonia classifier, but to study how an existing model's predictions and confidence behave on an external dataset.


In [46]:
class SmallCNN(nn.Module):
    """
    The same SmallCNN architecture used in Stage 1.

    Keeping the architecture unchanged is necessary because the
    saved Stage 1 weights belong to this exact network structure.
    """

    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(

            # Learn basic visual patterns from the one-channel X-ray.
            nn.Conv2d(
                in_channels=1,
                out_channels=16,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU(),

            # Reduce the spatial size while keeping the learned features.
            nn.MaxPool2d(2),


            # Learn more detailed visual patterns.
            nn.Conv2d(
                in_channels=16,
                out_channels=32,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU(),

            nn.MaxPool2d(2),


            # Learn higher-level image features.
            nn.Conv2d(
                in_channels=32,
                out_channels=64,
                kernel_size=3,
                padding=1
            ),

            nn.ReLU(),

            # Convert each feature map into one representative value.
            nn.AdaptiveAvgPool2d(1)
        )


        # Convert the 64 learned features into one binary classification logit.
        self.classifier = nn.Linear(64, 1)


    def forward(self, x):

        # Extract image features.
        x = self.features(x)

        # Flatten the feature representation.
        x = x.flatten(1)

        # Return the raw classification score.
        return self.classifier(x).squeeze(1)


## 13. Load the trained Stage 1 weights

The model is now recreated with the same architecture as Stage 1.

We load the saved weights and immediately switch the network to evaluation mode. No RSNA image is used to train or update the model in Stage 2.


In [47]:
# Use a GPU if one is available; otherwise use the CPU.
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


# Create the same CNN architecture used in Stage 1.
model = SmallCNN().to(device)


# Load the parameters learned during Stage 1.
state_dict = torch.load(
    checkpoint_path,
    map_location=device
)

model.load_state_dict(
    state_dict
)


# Evaluation mode tells PyTorch that we are only making predictions.
model.eval()


print("Stage 1 model loaded successfully.")
print("Using device:", device)


C:\Users\PRASAD\AppData\Local\Temp\ipykernel_31988\3564712263.py:12: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(


Stage 1 model loaded successfully.
Using device: cpu


## 14. Create the RSNA evaluation dataset

Only DICOM files that successfully received an RSNA ground-truth label are included.

The Dataset reads one image at a time, which avoids loading the complete RSNA image collection into memory.


In [48]:
# Keep only DICOM files that have a valid RSNA study-level label.
rsna_eval_df = dicom_df[
    dicom_df["label"].notna()
].copy()

# Reset the index so Dataset indexing starts cleanly from zero.
rsna_eval_df = rsna_eval_df.reset_index(drop=True)

print(
    "RSNA images available for evaluation:",
    len(rsna_eval_df)
)


class RSNAEvaluationDataset(Dataset):
    """
    Dataset used for external RSNA evaluation.

    Each item contains:
        - one preprocessed chest X-ray
        - its RSNA ground-truth label
    """

    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):

        row = self.dataframe.iloc[index]

        # Read the DICOM image for this particular sample.
        ds = pydicom.dcmread(
            row["dicom_path"]
        )

        # Convert the pixel data into a grayscale PIL image.
        image = Image.fromarray(
            ds.pixel_array,
            mode="L"
        )

        # Apply the Stage 1-compatible preprocessing.
        if self.transform is not None:
            image = self.transform(image)

        # Return both the image and the known RSNA label.
        label = int(row["label"])

        return image, label


RSNA images available for evaluation: 25684


## 15. Create the RSNA DataLoader

The DataLoader processes the external dataset in batches.

`shuffle=False` is intentional because we are evaluating the model, not training it. Keeping the order fixed also makes the prediction arrays reproducible.


In [49]:
# Create the Dataset that reads the RSNA images on demand.
rsna_dataset = RSNAEvaluationDataset(
    rsna_eval_df,
    transform=rsna_transform
)


# Process 128 images at a time.
# This keeps memory usage manageable while still allowing efficient inference.
rsna_loader = DataLoader(
    rsna_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=0
)


print("RSNA dataset size:", len(rsna_dataset))
print("RSNA batches:", len(rsna_loader))


RSNA dataset size: 25684
RSNA batches: 201


## 16. Run external inference

This is the main Stage 2 experiment.

For every RSNA image we collect:

- the raw model **logit**
- the model's **pneumonia probability**
- the known **RSNA ground-truth label**

The raw logits are retained because Stage 3 temperature scaling is mathematically applied to logits rather than directly modifying the probabilities.


In [50]:
# These lists temporarily hold one NumPy array per batch.
# Keeping the batches separate during inference avoids creating a
# very large array repeatedly while the model is running.
all_logits = []
all_probabilities = []
all_labels = []


# Make sure the model is in evaluation mode.
model.eval()


# We do not need gradients because the model is not being trained here.
with torch.no_grad():

    for batch_number, (images, labels) in enumerate(
        rsna_loader,
        start=1
    ):

        # Move the image batch to the same device as the model.
        images = images.to(device)

        # Get the raw model scores.
        logits = model(images)

        # Convert the raw scores into probabilities between 0 and 1.
        probabilities = torch.sigmoid(logits)


        # Move the results back to the CPU and store them as NumPy arrays.
        all_logits.append(
            logits.cpu().numpy()
        )

        all_probabilities.append(
            probabilities.cpu().numpy()
        )

        all_labels.append(
            labels.numpy()
        )


        # Print progress periodically rather than printing every batch.
        if (
            batch_number % 25 == 0
            or batch_number == len(rsna_loader)
        ):
            print(
                f"Processed batch "
                f"{batch_number:,} / {len(rsna_loader):,}"
            )


# Combine all batches into one continuous array.
all_logits = np.concatenate(all_logits)
all_probabilities = np.concatenate(all_probabilities)
all_labels = np.concatenate(all_labels)


print("\nRSNA inference completed.")
print("Predictions:", len(all_probabilities))
print("Labels:", len(all_labels))
print("Logits:", len(all_logits))


Processed batch 25 / 201
Processed batch 50 / 201
Processed batch 75 / 201
Processed batch 100 / 201
Processed batch 125 / 201
Processed batch 150 / 201
Processed batch 175 / 201
Processed batch 200 / 201
Processed batch 201 / 201

RSNA inference completed.
Predictions: 25684
Labels: 25684
Logits: 25684


## 17. Validate the prediction arrays

Before saving anything, we check that all three arrays have the same length and that the probabilities are within the expected 0 to 1 range.

For the current RSNA experiment, we expect the external prediction count to match the labelled DICOM count.


In [51]:
# All three arrays must refer to the same images in the same order.
if not (
    len(all_logits)
    == len(all_probabilities)
    == len(all_labels)
):
    raise ValueError(
        "Prediction arrays do not have the same length."
    )


# Probabilities produced by sigmoid must fall between 0 and 1.
if (
    np.min(all_probabilities) < 0
    or np.max(all_probabilities) > 1
):
    raise ValueError(
        "Found a probability outside the valid 0 to 1 range."
    )


print("Prediction arrays passed validation.")
print("Number of RSNA predictions:", len(all_probabilities))
print(
    "Minimum probability:",
    f"{all_probabilities.min():.6f}"
)
print(
    "Maximum probability:",
    f"{all_probabilities.max():.6f}"
)


print("\nGround-truth distribution:")
print(
    pd.Series(all_labels).value_counts().sort_index()
)


Prediction arrays passed validation.
Number of RSNA predictions: 25684
Minimum probability: 0.212479
Maximum probability: 0.974634

Ground-truth distribution:
0    20025
1     5659
Name: count, dtype: int64


## 18. Save the Stage 2 predictions

The predictions are saved separately from the raw DICOM data.

This is important for reproducibility because Stage 3 can now perform calibration analysis without running the entire RSNA inference process again.


In [52]:
# Make sure the research-data directory exists.
prediction_path.parent.mkdir(
    parents=True,
    exist_ok=True
)


# Save the raw logits, probabilities, and ground-truth labels together.
np.savez(
    prediction_path,
    logits=all_logits,
    probabilities=all_probabilities,
    labels=all_labels
)


print("Stage 2 prediction file saved successfully.")
print("Path:")
print(prediction_path)
print("\nFile exists:", prediction_path.exists())


Stage 2 prediction file saved successfully.
Path:
D:\0001_Kings_AI_Project\trustworthy-ai-medical-imaging\research_data\rsna_stage2_predictions.npz

File exists: True


# Stage 2 Findings: External Evaluation on the RSNA Dataset

Stage 2 was designed to move beyond the controlled image perturbation used in Stage 1 and test whether the same trained model behaves differently when evaluated on an independently sourced chest X-ray dataset.

The important point here is that the model was **not retrained on the RSNA dataset**. The SmallCNN trained during Stage 1 was loaded directly from the saved checkpoint and used as-is. This means that any change in behaviour during this experiment comes from the difference between the original training environment and the external dataset, rather than from additional training on RSNA images.

### Dataset construction and validation

The RSNA dataset contained 25,684 studies with 28,989 individual annotations.

Some studies had more than one annotation, so the number of annotations was larger than the number of studies. After grouping the annotations at study level, the dataset contained:

- 20,025 studies labelled as `No Lung Opacity`
- 5,659 studies labelled as `Lung Opacity`
- 25,684 unique studies in total

The study-level label table contained no duplicate StudyInstanceUID values and no missing labels.

The DICOM data were also checked before inference. A total of 25,684 DICOM images were found, with one matching image for each labelled study. All 25,684 images had a matching RSNA label, with no unmatched images, duplicate SOPInstanceUID values, or missing StudyInstanceUID values.

The sample DICOM files were 1024 × 1024 grayscale images using MONOCHROME2 encoding. After preprocessing, the images were converted to the same 64 × 64 single-channel format expected by the Stage 1 model.

These checks are important because they give us confidence that the external evaluation was performed on a complete and correctly matched set of images and labels.

### Running the Stage 1 model on RSNA

The trained Stage 1 checkpoint was successfully loaded and used for inference on all 25,684 RSNA images.

The inference produced:

- 25,684 logits
- 25,684 probability predictions
- 25,684 ground-truth labels

The number of predictions exactly matched the number of available labelled images, and the validation checks passed.

The predicted probabilities ranged from approximately 0.212 to 0.975. This shows that the model produced a reasonable range of confidence values across the external dataset, although the probability range itself should not be interpreted as evidence that the model is well calibrated.

### An important difference in the labels

There is an important distinction between the Stage 1 and Stage 2 tasks.

The Stage 1 model was trained using PneumoniaMNIST, where the target represents a pneumonia classification task. The RSNA dataset used in Stage 2 instead provides study-level labels based on the presence or absence of lung opacity.

These labels are related, but they are not identical clinical definitions.

Therefore, this experiment should not be described simply as testing "pneumonia diagnosis on RSNA". A more accurate description is:

> The Stage 2 experiment evaluates a PneumoniaMNIST-trained model on an external chest X-ray dataset using RSNA study-level lung-opacity labels.

This distinction matters because differences between the two label definitions may contribute to the model's external performance.

### What Stage 2 establishes

The main purpose of this stage was to establish a clean external evaluation pipeline.

The complete dataset could be loaded, the study-level labels could be constructed, the DICOM images could be matched to those labels, and the original Stage 1 model could make predictions for the entire external dataset without retraining.

This gives us a controlled setup for studying external distribution shift.

The external dataset also introduces differences that are more representative of real-world variation than the artificial contrast experiment in Stage 1. The images come from a different dataset and therefore may differ in image characteristics, acquisition conditions, patient population, labelling process, and other factors.

However, Stage 2 by itself does not tell us whether the model is robust or poorly calibrated. That requires analysing the predictions using classification and probability-based metrics.

### Limitation of the external evaluation

The RSNA dataset is useful for evaluating external behaviour, but it should not be treated as a perfect representation of hospital-to-hospital clinical deployment.

The experiment also uses a model trained on PneumoniaMNIST and evaluates it against RSNA lung-opacity labels without retraining or adapting the model. Consequently, any performance difference may reflect a combination of dataset shift, differences in label definitions, image characteristics, and the limitations of the small baseline model.

For this reason, the external results should be interpreted as evidence of **cross-dataset behaviour**, rather than as a direct estimate of clinical performance.

### Next step

The predictions generated in this stage provide the basis for the next experiment.

In Stage 3, we will examine whether temperature scaling can improve the reliability of the model's probability estimates on the external RSNA data.

The goal is not simply to increase classification accuracy. Instead, we want to investigate whether the model can become better calibrated, so that a predicted probability is more closely aligned with the observed frequency of correct predictions.

This moves the experiment from asking:

**"Is the model correct?"**

towards the more important question for trustworthy AI:

**"When the model is confident, can we trust that confidence?"**